# Init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col, trim
from pyspark.sql.types import StringType
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
RENAME_MAP = {
    "CID": "customer_key",
    "BDATE": "birthdate",
    "GEN": "gender"
}

# Reading from Bronze layer

In [0]:
df = spark.table("data_lakehouse_project.bronze.erp_cust_az12")

# Data Transformation

- TRIM the strings
- Column CID UPPER + Remove "NAS"
- Managed GEN column, "NULL" + Standardization
- Rename Columns

In [0]:
df.display()

## Triming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

## Upper CID column + Standardized to match other table

In [0]:
df = df.withColumn("CID", F.upper(col("CID")))

In [0]:
df = df.withColumn("CID", F.regexp_replace(F.col("CID"), "-|NAS", ""))

## GEN column

In [0]:
df = df.withColumn(
    "GEN",
    F.when(F.col("GEN").rlike("^[Ff]"), "Female")
     .when(F.col("GEN").rlike("^[Mm]"), "Male")
     .otherwise("N/A")
)

## Rename Columns

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Write into Silver Table

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("data_lakehouse_project.silver.erp_cust_az12")
)

# Check queries

In [0]:
%sql
SELECT *
FROM data_lakehouse_project.silver.erp_cust_az12
